# Capstone : Content Refresh / Opportunity Scoring

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This notebook is the end-to-end summary of Weeks 1–7. It re-derives nothing from scratch — it reads the artifacts each weekly notebook already produced and presents them in the paper's order. Full derivations live in `w01`–`w07`.

## 1. Question

**Lane:** Content Refresh / Opportunity Scoring.

**Research question:** Can a model trained on observable 90-day search performance rank pages by refresh urgency more precisely, at the top of the queue, than a straightforward age-and-CTR rule — under a validation split that can't leak client-specific patterns?

**Decision it supports:** which pages a content editor opens first this week, under a fixed weekly review capacity. **Cost of a wrong call:** a false positive wastes a writer's hours on a healthy page; a false negative lets a decaying page keep losing traffic to competitors, unnoticed.

## 2. Data

*Which release, which tables, date windows, what was excluded and why.*

In [8]:
# Data contract summary (full derivation + verification queries: w03_data_contract.ipynb)
print("Release: fact_content_daily_performance, month=2026-03")
print("Full population: 331,437 unique content items | 9,841,378 daily rows")
print("Working sample: 30,000 content assets across 32 clients")
print("Filter: impressions_90d > 0 AND content_age_days >= 90")
print()
print("Features (5): impressions_90d, clicks_90d, ctr_decimal, avg_position_clean, sessions_90d")
print("Label: target_needs_refresh (constructed proxy, not hand-labeled ground truth)")
print()
print("Excluded:")
print(" - trend_direction, trend_pct  -> define the label; including them would be leakage")
print(" - client_id, content_id      -> used only for split grouping, never as features")
print(" - long-tail queries          -> redacted by GSC privacy thresholding")

Release: fact_content_daily_performance, month=2026-03
Full population: 331,437 unique content items | 9,841,378 daily rows
Working sample: 30,000 content assets across 32 clients
Filter: impressions_90d > 0 AND content_age_days >= 90

Features (5): impressions_90d, clicks_90d, ctr_decimal, avg_position_clean, sessions_90d
Label: target_needs_refresh (constructed proxy, not hand-labeled ground truth)

Excluded:
 - trend_direction, trend_pct  -> define the label; including them would be leakage
 - client_id, content_id      -> used only for split grouping, never as features
 - long-tail queries          -> redacted by GSC privacy thresholding


## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

In [9]:
# Methodology summary (full derivation: w04_baseline_score.ipynb, w05_model.ipynb, w06_validation_audit.ipynb)
print("Baseline: rule -- age > 180d AND ctr < 2% despite meaningful views")
print("Models compared: Logistic Regression, Random Forest")
print("Split: GroupShuffleSplit by client_id, 80/20, random_state=42")
print("  Train: 23,837 rows (35.5% positive) | Test: 6,163 rows (24.5% positive)")
print()
print("Leakage checks run on the honest split:")
print("  1. Target-source columns (trend_direction, trend_pct) confirmed absent from feature matrix -- PASS")
print("  2. Feature-target correlation audit -- no feature abnormally high")
print("  3. Permutation importance -- top feature (impressions_90d) = 0.224, below 0.40 leakage threshold -- PASS")
print()
print("Key finding: the SAME Random Forest scored 95% Precision@20 under a naive row-level random split,")
print("vs 70% under the honest client-grouped split -- a 25-point gap caused entirely by client identity")
print("leaking across train/test in the naive version.")

Baseline: rule -- age > 180d AND ctr < 2% despite meaningful views
Models compared: Logistic Regression, Random Forest
Split: GroupShuffleSplit by client_id, 80/20, random_state=42
  Train: 23,837 rows (35.5% positive) | Test: 6,163 rows (24.5% positive)

Leakage checks run on the honest split:
  1. Target-source columns (trend_direction, trend_pct) confirmed absent from feature matrix -- PASS
  2. Feature-target correlation audit -- no feature abnormally high
  3. Permutation importance -- top feature (impressions_90d) = 0.224, below 0.40 leakage threshold -- PASS

Key finding: the SAME Random Forest scored 95% Precision@20 under a naive row-level random split,
vs 70% under the honest client-grouped split -- a 25-point gap caused entirely by client identity
leaking across train/test in the naive version.


## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

In [10]:
import pandas as pd

# Results, honest client-grouped test set (n=6,163, 24.5% base rate)
# Full derivation: w05_model.ipynb section 3, cross-checked in w06_validation_audit.ipynb section 2
results = pd.DataFrame({
    "Method": ["Random selection", "Rule baseline", "Logistic Regression", "Random Forest"],
    "Precision@20": ["24.5%", "20.0%", "40.0%", "70.0%"],
    "Precision@50": ["24.5%", "34.0%", "38.0%", "68.0%"],
    "PR-AUC": [0.245, 0.242, 0.258, 0.583]
})
print(results.to_string(index=False))
print()
print("Note: rule baseline scores BELOW random at K=20 (20.0% vs 24.5%) -- reported as-is,")
print("not smoothed over. It recovers to 34% by K=50.")
print()
print("Top feature (permutation importance): impressions_90d (0.224)")
print("Efficiency gain from using the ranked queue over unordered review: 2.86x")

             Method Precision@20 Precision@50  PR-AUC
   Random selection        24.5%        24.5%   0.245
      Rule baseline        20.0%        34.0%   0.242
Logistic Regression        40.0%        38.0%   0.258
      Random Forest        70.0%        68.0%   0.583

Note: rule baseline scores BELOW random at K=20 (20.0% vs 24.5%) -- reported as-is,
not smoothed over. It recovers to 34% by K=50.

Top feature (permutation importance): impressions_90d (0.224)
Efficiency gain from using the ranked queue over unordered review: 2.86x


## 5. Limitations

*What this work cannot claim.*

In [11]:
limitations = [
    "Not causal -- ranks by observed decay signal, does not prove refreshing a page recovers its traffic",
    "Single 30-day window (2026-03) -- seasonal decay patterns are not visible to the model",
    "GSC privacy thresholding under-reports long-tail query volume",
    "Visibility-only signal -- no on-page conversion or revenue data included",
    "target_needs_refresh is a constructed proxy label, not a hand-audited editorial judgment"
]
for i, l in enumerate(limitations, 1):
    print(f"{i}. {l}")

1. Not causal -- ranks by observed decay signal, does not prove refreshing a page recovers its traffic
2. Single 30-day window (2026-03) -- seasonal decay patterns are not visible to the model
3. GSC privacy thresholding under-reports long-tail query volume
4. Visibility-only signal -- no on-page conversion or revenue data included
5. target_needs_refresh is a constructed proxy label, not a hand-audited editorial judgment


## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

In [12]:
import json, os

# Reads the artifact already exported in w07_action_playbook.ipynb
path = "../metrics/playbook_metrics.json"
if os.path.exists(path):
    with open(path) as f:
        print(json.dumps(json.load(f), indent=2))
else:
    print("playbook_metrics.json not found at", path, "-- run w07_action_playbook.ipynb first.")
    print()
    print("Summary: 4 archetypes (RC_FULL_REFRESH, STALE_LOW_CTR, STALE_CONTENT, MONITOR).")
    print("15,086 pages flagged for mandatory human sign-off; 14,914 in standard monitoring.")
    print("Strict automation ban: no auto-deletion, no unreviewed auto-publishing.")
    print("Retrain: quarterly, or immediately if rolling Precision@20 drops below 50%.")

{
  "model_version": "w06_rf_honest_v1",
  "evaluation_metrics": {
    "test_base_rate": 0.245,
    "precision_at_20": 0.7,
    "precision_at_50": 0.68
  },
  "retrain_triggers": {
    "precision_at_20_min_threshold": 0.5,
    "max_data_age_days": 90,
    "algo_update_wait_days": 14
  },
  "no_go_policy_enforced": true
}


## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

In [13]:
import os

artifacts = [
    "../outputs/ranked_action_queue.csv",
    "../outputs/baseline_action_score.csv",
    "../metrics/playbook_metrics.json",
    "../figures/action_archetype_distribution.png",
]
for a in artifacts:
    print(("[OK] " if os.path.exists(a) else "[MISSING] ") + a)
print()
print("These feed the paper's Results and Recommendations sections directly.")
print("Deployed paper: see submission/paper_url.txt")

[OK] ../outputs/ranked_action_queue.csv
[OK] ../outputs/baseline_action_score.csv
[OK] ../metrics/playbook_metrics.json
[OK] ../figures/action_archetype_distribution.png

These feed the paper's Results and Recommendations sections directly.
Deployed paper: see submission/paper_url.txt


## 8. 5-minute demo outline

*For the Week-8 showcase, if presenting. Question -> method -> one chart -> one honest result -> one recommendation.*

**0:00 - 0:45 — Question.** Content editors can't manually audit thousands of pages a week to find what needs refreshing. Can a model rank pages by refresh urgency more precisely than a simple age-and-CTR rule, without leaking client-specific patterns into the score?

**0:45 - 2:00 — Method.** 30,000 content assets across 32 clients, 5 decision-time-knowable features (impressions, clicks, CTR, position, sessions). Random Forest vs. a transparent rule baseline, evaluated on a `client_id`-grouped 80/20 split so no client's pages cross between train and test.

**2:00 - 3:00 — One chart.** The Precision@20 bar comparison from the paper's Results section: Random (24.5%) -> Rule (20.0%) -> Logistic Regression (40.0%) -> Random Forest (70.0%).

**3:00 - 4:00 — One honest result.** The split mattered more than the model choice: the same Random Forest scored 95% Precision@20 under a naive random split and 70% under the honest client-grouped split — a 25-point leakage artifact, caught and corrected, not hidden.

**4:00 - 5:00 — One recommendation.** Editors work a ranked queue by reason code (`RC_FULL_REFRESH`, `STALE_LOW_CTR`, etc.) instead of a flat list — about a 2.9x gain in how many top-priority picks actually needed attention, with mandatory human sign-off on the highest-impact pages.

## 9. Shareable cuts

**Social post (methodology):**

> Spent my ML internship capstone on a boring-sounding but real problem: which of 30,000 web pages actually need a content refresh? Built a rule baseline first (age + CTR), then a Random Forest on 5 features. The twist: when I evaluated with a naive random split, my model looked amazing — 95% precision in its top 20 picks. Grouped the split by client instead (so no client's pages touch both train and test) and that dropped to a real 70%. Still a 2.9x lift over random review, just an honest one. The validation strategy mattered more than the model.

**Employer-facing summary (3 sentences):**

I built a Random Forest model that ranks web pages by content-refresh urgency, trained on 30,000 pages across 32 clients using five observable search-performance signals (impressions, clicks, CTR, position, sessions). Validated on a client-grouped holdout split to prevent leakage, it achieved 70% precision in its top 20 recommendations versus 20% for a rule-based baseline and a 24.5% base rate — roughly a 2.9x gain in editorial efficiency. The project also surfaced and corrected a 25-point leakage artifact from an initial naive validation split, and ships as a decision-support ranking tool, not a causal claim.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all) — **run this yourself once `work/` artifacts exist in your environment**
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [x] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [x] **ML-12** (5-minute demo outline + social-post cut + 3-sentence employer summary) — sections 8 and 9 above.
